# Finding Redundant 1D Subspaces in LLM Activations

This notebook discovers **redundant 1-dimensional subspaces** in language model activations: directions where:
1. Activations have **large projections** (model actively uses this direction)
2. **Removing** this projection has **minimal impact** on output token distributions

In [1]:
from src.utils.env import set_seed

set_seed(42)

In [ ]:
import torch
from src.load.load_model import load_model

torch.set_float32_matmul_precision("high")

model, tokenizer = load_model("google/gemma-2b-it", torch_dtype="bfloat16")

In [ ]:
model

In [ ]:
from src.utils.logging import create_logger

logger = create_logger(__name__)

In [ ]:
from transformers.tokenization_utils_base import BatchEncoding
from transformers import PreTrainedTokenizer
from src.aliases import Conv
from tqdm import tqdm
from src.model_helpers import tokenize

In [ ]:
from typing import Any
import numpy as np


def split_data(data: list[Any], train_ratio: float = 0.6, val_ratio: float = 0.2, seed: int = 42) -> dict[str, list[Any]]:
    assert 0 < train_ratio < 1, "train_ratio must be between 0 and 1"
    assert 0 < val_ratio < 1, "val_ratio must be between 0 and 1"
    assert 0 < train_ratio + val_ratio < 1, "train_ratio + val_ratio must be between 0 and 1"

    rng = np.random.default_rng(seed)
    shuffle_indices = rng.permutation(len(data))
    data = [data[i] for i in shuffle_indices]

    total_count = len(data)
    num_train = int(total_count * train_ratio)
    num_val = int(total_count * val_ratio)

    train_data = data[:num_train]
    val_data = data[num_train : num_train + num_val]
    test_data = data[num_train + num_val :]

    logger.info(f"Total data points: {len(data)}")
    logger.info(f"Training data points: {len(train_data)}")
    logger.info(f"Validation data points: {len(val_data)}")
    logger.info(f"Test data points: {len(test_data)}")

    return {"train": train_data, "val": val_data, "test": test_data}

In [ ]:
from src.load.load_dataset import load_dataset

ds_train, ds_val, ds_test = load_dataset("hh-rlhf")

train_convs = list(map(lambda conv: conv.tolist(), ds_train["chosen"].to_list()))
convs = list(map(lambda conv: conv.tolist(), ds_val["chosen"].to_list()))
test_convs = list(map(lambda conv: conv.tolist(), ds_test["chosen"].to_list()))

In [ ]:
from src.losses import compute_targets_mask
from src.model_helpers import project

In [ ]:

import matplotlib.pyplot as plt
from transformers.tokenization_utils_base import BatchEncoding
from transformers.tokenization_utils import PreTrainedTokenizer
from transformers import PreTrainedModel
from src.activation_extractor import ActivationExtractor
from src.utils.torch import extract_device


class DisplayFormatter:
    """Centralized formatting for console output."""

    SECTION_WIDTH = 80
    WIDE_WIDTH = 120

    @staticmethod
    def section_header(title: str, width: int = SECTION_WIDTH) -> str:
        return f"\n{'=' * width}\n{title}\n{'=' * width}\n"

    @staticmethod
    def subsection(title: str, width: int = SECTION_WIDTH) -> str:
        return f"\n{'-' * width}\n{title}\n{'-' * width}"

    @staticmethod
    def metric_line(label: str, value: float, unit: str = "", precision: int = 4) -> str:
        fmt = f"{{value:.{precision}f}}"
        value_str = fmt.format(value=value)
        return f"  {label:.<40} {value_str}{unit}"


@torch.inference_mode()
def plot_conversation_norms(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    conv: Conv,
    layer_name: str,
    direction: torch.Tensor,
    ax: plt.Axes | None = None,
    title_suffix: str = "",
):
    device = extract_device(model)
    activ_extractor = ActivationExtractor(model, layer_name)

    encodings = tokenize([conv], tokenizer).to(device)

    with activ_extractor.capture():
        model(**encodings)
        activs = activ_extractor.get_activations()[layer_name]
        
    # compute normalized projections
    activs = torch.nn.functional.normalize(activs, dim=-1)
    projections = project(activs, direction=direction, normalize=True)
    l2_norms = torch.norm(projections, dim=-1)

    targets_mask = compute_targets_mask(encodings)
    token_list = encodings.input_ids[targets_mask.bool()].tolist()
    norms_list = l2_norms[targets_mask.bool()].tolist()
    token_strs = [tokenizer.decode([tid]) for tid in token_list]

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(15, 3))
        
    x_pos = np.arange(len(token_strs))
    ax.bar(x_pos, norms_list, alpha=0.7)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(token_strs, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("L2 Norm")
    ax.set_title(f"Prompt: Projection Magnitudes {title_suffix}")
    ax.grid(axis="y", alpha=0.3)


# Multi-Location Redundant Subspace Search

In [ ]:
from math import floor


def get_search_locations(
    num_probes=10,
    block_path: str | None = "model.layers.{i}",
    attn_path: str | None = "model.layers.{i}.self_attn",
    mlp_path: str | None = "model.layers.{i}.mlp",
) -> list[tuple[str, str, str]]:
    """
    Define architecturally meaningful locations to search for redundant directions.
    """
    num_hidden_layers = model.config.num_hidden_layers
    num_probes = min(num_probes, num_hidden_layers)

    indices = [floor(i * num_hidden_layers / num_probes) for i in range(num_probes)]
    if num_hidden_layers - 1 not in indices:
        indices.append(num_hidden_layers - 1)

    locations = []

    if block_path is not None:
        locations += [(f"residual_{i}", block_path.format(i=i), f"Residual stream after block {i}") for i in indices]

    if attn_path is not None:
        locations += [(f"attn_output_{i}", attn_path.format(i=i), f"Attention output after block {i}") for i in indices]

    if mlp_path is not None:
        locations += [(f"mlp_output_{i}", mlp_path.format(i=i), f"MLP output after block {i}") for i in indices]

    special_locations = [
        ("embeddings", "model.embed_tokens", "Token embeddings (before any layers)"),
        ("final_norm", "model.norm", "Final normalization (before lm_head)"),
    ]

    locations += special_locations

    print("=" * 80)
    print(f"REDUNDANCY EXPLORATION: {len(locations)} LOCATIONS")
    print("=" * 80)
    print(f"\n{'Name':<30} {'Path':<50} Description")
    print("=" * 80)
    for name, path, desc in locations:
        print(f"  {name:<30} {path:<50} {desc}")

    # return locations
    return [locations[0]]


# Get search locations
search_locations = get_search_locations()

In [ ]:
import time
from transformers import PreTrainedModel, PreTrainedTokenizer
from src.train import validate_redundant_subspace, find_redundant_1d_subspace


def search_redundant_directions_multi_location(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    search_locations: list[tuple[str, str, str]],
    train_convs: list[Conv],
    val_convs: list[Conv],
    num_iterations: int = 500,
    lr: float = 0.01,
    projection_weight: float = 0.1,
    batch_size: int = 8,
) -> dict[str, dict]:
    """
    Search for redundant 1D subspaces across multiple network locations.
    """
    location_results = {}

    print("=" * 80)
    print("SEARCHING FOR REDUNDANT 1D SUBSPACES ACROSS MULTIPLE LOCATIONS")
    print("=" * 80)
    print(f"  Training set:   {len(train_convs)} conversations")
    print(f"  Validation set: {len(val_convs)} conversations (completely separate)")
    print(f"Batch size: {batch_size} (used for both optimization and validation)")

    for location_name, layer_path, description in search_locations:
        print(f"\n{'=' * 80}")
        print(f"LOCATION: {location_name}")
        print(f"Path: {layer_path}")
        print(f"Description: {description}")
        print(f"{'=' * 80}\n")

        start_time = time.time()

        # Find redundant direction at this location using TRAINING SET ONLY
        redundant_dir = find_redundant_1d_subspace(
            model=model,
            tokenizer=tokenizer,
            layer_name=layer_path,
            conversations=train_convs,  # Use TRAINING set for optimization
            num_iterations=num_iterations,
            lr=lr,
            projection_weight=projection_weight,
            batch_size=batch_size,
        )

        # Validate on the full VALIDATION SET (completely separate from training)
        print(f"\nValidating on {len(val_convs)} validation samples (separate from training)...")
        val_metrics = validate_redundant_subspace(
            model=model,
            tokenizer=tokenizer,
            layer_name=layer_path,
            convs=val_convs,
            redundant_dir=redundant_dir,
            batch_size=batch_size,
        )

        elapsed = time.time() - start_time

        # Store results
        location_results[location_name] = {
            "layer_path": layer_path,
            "description": description,
            "direction": redundant_dir,
            "metrics": val_metrics,
            "time_sec": elapsed,
        }

        print(f"\n✓ Completed in {elapsed:.1f}s")
        print(f"  Redundancy Score:           {val_metrics['redundancy_score']:.4f}")
        print(f"  Top-1 Accuracy:             {val_metrics['top1_accuracy']:.4f}")
        print(f"  Top-10 Agreement:           {val_metrics['top10_agreement']:.4f}")
        print(f"  KL Divergence:              {val_metrics['kl_divergence']:.6f}")
        print(f"  Raw Projection L2:          {val_metrics['projection_mean_raw']:.4f}")
        print(f"  Normalized Projection L2:   {val_metrics['projection_mean_normalized']:.4f} ⭐")

    print(f"\n{'=' * 80}")
    print("ALL LOCATIONS SEARCHED")
    print(f"{'=' * 80}")

    return location_results


In [ ]:
# Run the search across all locations
location_results = search_redundant_directions_multi_location(
    model=model,
    tokenizer=tokenizer,
    search_locations=search_locations,
    train_convs=train_convs,
    val_convs=convs,
    num_iterations=30,
    lr=0.01,
    projection_weight=0.1,
    batch_size=6,
)

In [ ]:
import pandas as pd


def _create_comparison_dataframe(location_results: dict[str, dict]) -> pd.DataFrame:
    """Build comparison dataframe from location results."""
    data = []
    for name, result in location_results.items():
        m = result["metrics"]
        data.append(
            {
                "Location": name,
                "Description": result["description"],
                "Redundancy": f"{m['redundancy_score']:.4f}",
                "Top-1 Acc": f"{m['top1_accuracy']:.4f}",
                "Top-10 Agr": f"{m['top10_agreement']:.4f}",
                "KL-Div": f"{m['kl_divergence']:.6f}",
                "Proj.Raw": f"{m['projection_mean_raw']:.4f}",
                "Proj.Norm": f"{m['projection_mean_normalized']:.4f}",
                "Time(s)": f"{result['time_sec']:.1f}",
            }
        )
    return pd.DataFrame(data)


def _print_location_insight(title: str, location: tuple[str, dict], metric_name: str = None, metric_value: float = None) -> None:
    """Print a formatted insight about a specific location."""
    print(f"\n{title}:")
    print(f"   Location: {location[0]}")
    print(f"   {location[1]['description']}")
    m = location[1]["metrics"]
    print(f"   Redundancy Score: {m['redundancy_score']:.4f}")
    print(f"   Top-1 Accuracy: {m['top1_accuracy']:.4f}")
    print(f"   Normalized Proj: {m['projection_mean_normalized']:.4f} ({m['projection_mean_normalized'] * 100:.1f}%)")
    if metric_name and metric_value is not None:
        print(f"   {metric_name}: {metric_value:.6f}")


def analyze_and_compare_results(location_results: dict[str, dict]) -> pd.DataFrame:
    print(DisplayFormatter.section_header("COMPARISON: REDUNDANT SUBSPACES ACROSS LOCATIONS", DisplayFormatter.WIDE_WIDTH))

    df_comparison = _create_comparison_dataframe(location_results)
    print(df_comparison.to_string(index=False))

    print("\nMetrics Legend:")
    print("  • Redundancy = Canonical Metric used to compute overall direction score")
    print("  • Top-1 Acc = Fraction where top-1 token unchanged after ablation")
    print("  • Top-10 Agr = Fraction of top-10 tokens that overlap after ablation")
    print("  • KL-Div = Distribution divergence (lower = better preserved)")
    print("  • Proj.Norm = Normalized projection magnitude [0-1] (proportional importance) ⭐")

    print(DisplayFormatter.section_header("KEY INSIGHTS"))

    best_redundancy = max(location_results.items(), key=lambda x: x[1]["metrics"]["redundancy_score"])
    _print_location_insight("1. STRONGEST REDUNDANCY", best_redundancy)

    best_kl = min(location_results.items(), key=lambda x: x[1]["metrics"]["kl_divergence"])
    _print_location_insight("2. BEST DISTRIBUTION PRESERVATION", best_kl, "KL Divergence", best_kl[1]["metrics"]["kl_divergence"])

    best_proj = max(location_results.items(), key=lambda x: x[1]["metrics"]["projection_mean_normalized"])
    _print_location_insight("3. LARGEST PROPORTIONAL PROJECTIONS", best_proj)

    return df_comparison


df_comparison = analyze_and_compare_results(location_results)


In [ ]:
import random
import matplotlib.pyplot as plt

@torch.inference_mode()
def visualize_top_locations(
    location_results: dict[str, dict],
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    convs: list[Conv],
) -> None:
    """
    Visualize projection patterns for top N locations by redundancy score
    """
    sorted_locations = sorted(location_results.items(), key=lambda x: x[1]["metrics"]["redundancy_score"], reverse=True)

    num_convs = min(len(convs), 10)
    convs = random.sample(convs, num_convs)

    for rank, (location_name, result) in enumerate(sorted_locations, 1):
        print(DisplayFormatter.subsection(f"RANK {rank}: {location_name}"))
        print(f"{result['description']}")
        print(f"Redundancy Score: {result['metrics']['redundancy_score']:.4f}\n")

        layer_path = result["layer_path"]
        redundant_dir = result["direction"]
        
        fig, axes = plt.subplots(nrows=num_convs, ncols=1, figsize=(15, 3 * num_convs))

        for i, conv in enumerate():
            plot_conversation_norms(
                model=model,
                tokenizer=tokenizer,
                conv=conv,
                layer_name=layer_path,
                direction=redundant_dir,
                title_suffix=f"[{location_name}]",
                ax=axes[i]
            )


visualize_top_locations(
    location_results=location_results,
    model=model,
    tokenizer=tokenizer,
    convs=convs,
    top_n=3,
    num_vis_samples=5,
    num_prompts_per_location=3,
)


# Long-Term Generation Test

Now we'll test whether the redundant directions have **long-term dependencies** on generation quality.

Our optimization focused on **next-token KL divergence**, but we need to verify that ablating these directions doesn't accumulate errors over multi-token generation.

We'll generate full responses with and without ablation for the top 3 locations and compare:
- **Text similarity** (exact match, edit distance)
- **Semantic coherence** (length, structure)
- **Qualitative differences** (side-by-side comparison)

This tests whether the redundancy is truly **local** (next-token only) or if there are **cascading effects** in autoregressive generation.

In [ ]:
from src.model_helpers import generate, generate_ablated


@torch.inference_mode()
def test_generation_with_ablation(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    sorted_locations: list[tuple[str, dict]],
    test_convs: list[Conv],
    max_new_tokens: int = 100,
) -> dict[str, dict]:
    """
    Test long-term generation with and without direction ablation at top redundant locations.
    """
    print("\n" + "=" * 80)
    print("LONG-TERM GENERATION TEST: TOP REDUNDANT LOCATIONS")
    print("=" * 80)

    for rank, (location_name, result) in enumerate(sorted_locations, 1):
        print(f"\n{'=' * 80}")
        print(f"TESTING LOCATION {rank}: {location_name}")
        print(f"Description: {result['description']}")
        print(f"{'=' * 80}\n")

        layer_path = result["layer_path"]
        redundant_dir = result["direction"]

        # Generate baseline (no ablation)
        print("Generating baseline outputs (NO ablation)...")

        baseline_outputs = generate(
            model=model,
            tokenizer=tokenizer,
            prompts=test_convs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

        # Generate with ablation
        print("Generating ablated outputs (WITH ablation)...")

        ablated_outputs = generate_ablated(
            model=model,
            tokenizer=tokenizer,
            layer_name=layer_path,
            direction=redundant_dir,
            prompts=test_convs,
            max_new_tokens=max_new_tokens,
        )

        # now for each prompt, print the original baseline and ablated outputs side by side

        for i, conv in enumerate(test_convs):
            print(f"\nPrompt {i + 1}:")
            user_message = conv[0]["content"]
            print(f"User: {user_message}\n")

            baseline_response = baseline_outputs[i].strip()
            ablated_response = ablated_outputs[i].strip()

            print(f"Baseline Response:\n{baseline_response}\n")
            print(f"Ablated Response:\n{ablated_response}\n")
            print("-" * 80)


In [ ]:
# Test generation on top 3 locations

sorted_locations = [(k, res) for k, res in location_results.items()]
sorted_locations = sorted(sorted_locations, key=lambda item: item[1]["metrics"]["redundancy_score"], reverse=True)[:3]

generation_results = test_generation_with_ablation(
    model=model,
    tokenizer=tokenizer,
    sorted_locations=sorted_locations,
    test_convs=test_convs[:10],
    max_new_tokens=100,
)